# 🎓 Study Plan Supervisor

> An intelligent advisor that guides students on which courses to take next based on their academic history.

## 1. Project Description

University study plans are often complex and difficult to navigate. Many students struggle to identify the correct sequence of courses, which can lead to confusion and potential delays in graduation. This project solves that problem.

The **Study Plan Supervisor** acts as an AI-powered academic advisor. It analyzes a student's completed courses and provides clear, accurate recommendations for their next semester. This ensures that all prerequisites are met and that students have a clear, logical path toward completing their degree.

### v1.0 Scope

This initial version of the project is trained on the official study plans for three technical majors:
* Artificial Intelligence (AI)
* Computer Science (CS)
* Cybersecurity (CYS)

## 2. The Data Model

The model's intelligence is built on a structured understanding of course curricula. The training data includes the following key features for each course:

* `course_id`
* `dept_id`
* `course_name`
* `course_desc`
* `prerequisite_name`
* `prerequisite_number`
* `requirement_type` (e.g., University Requirement, Major Requirement)
* `number_of_hours`

### Rationale: Why This Data?

This specific data structure is the **reason** the advisor works.

* By linking a `course_id` with its `prerequisite_number`, the model can build a **dependency graph**. It doesn't just see a list of courses; it understands the *relationships* between them (e.g., "Data Structures" must be taken before "Algorithms").
* The `requirement_type` allows the model to prioritize essential courses (like major requirements) over others (like general electives), guiding the student more efficiently.

## 📚 Downloading Required Dependencies

In [ ]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.8/348.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.7/276.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [ ]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from unsloth import is_bfloat16_supported
import torch

from transformers import TrainingArguments, DataCollatorForSeq2Seq, AutoTokenizer

from datasets import load_dataset
from trl import SFTTrainer

### 3. 🛠️ Technology Stack & Rationale

This project is built on a modern, high-performance stack specifically chosen for fine-tuning Large Language Models (LLMs) efficiently.

* **`unsloth`**
    * **What it is:** A specialized library designed to dramatically speed up the fine-tuning of LLMs (like Llama) and significantly reduce memory usage.
    * **Reason for Use:** This is the core of the project's efficiency. Standard fine-tuning is extremely resource-intensive. **Unsloth** makes it possible to train this model on consumer-grade hardware (like in a Google Colab notebook) by optimizing the training process, allowing for faster development and iteration.

* **`torch` (PyTorch)**
    * **What it is:** A foundational, open-source machine learning library. It provides the "engine" for all mathematical operations and data structures (tensors) needed for deep learning.
    * **Reason for Use:** It is the required backend for the entire ecosystem. `transformers`, `unsloth`, and `trl` all run on top of PyTorch. It provides the low-level tools for calculating gradients and updating model weights during training.

* **`transformers` (Hugging Face)**
    * **What it is:** The standard library for working with pre-trained models. It provides the tools to download models, prepare data (like `AutoTokenizer`), and configure the training process.
    * **Reason for Use:** We are not building a model from scratch. We use `transformers` to load a powerful, pre-trained base model. The `TrainingArguments` class is used to set all key training parameters (like learning rate, batch size, and save-steps) in a simple, declarative way.

* **`datasets` (Hugging Face)**
    * **What it is:** A lightweight library for loading and processing large datasets efficiently.
    * **Reason for Use:** This is the simplest way to get our training data (the study plan JSON files) into the correct format. `load_dataset` handles all the "boring" work of reading files, parsing them, and making them available to the trainer.

* **`trl` (Transformer Reinforcement Learning)**
    * **What it is:** A library from Hugging Face for simplifying the process of fine-tuning models.
    * **Reason for Use:** We use a specific tool from this library: `SFTTrainer` (Supervised Fine-tuning Trainer). This trainer is a high-level wrapper that automates the entire training loop. We simply give it the model, the dataset, and the training arguments, and it handles all the complexity of feeding data to the model and saving the results.

## ⚙️ Model & Parameters Configuration

In [ ]:
# The maximum number of tokens in a single training example.
max_sequence = 2048

# Let the model automatically decide the best data type (e.g., float16).
dtype = None

# Load the model and tokenizer using Unsloth's optimized function.
load_in_4bit = True

In [ ]:
model = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit" # Base Model

model, tokenizer = FastLanguageModel.from_pretrained(
    model,
    max_seq_length=max_sequence,
    dtype=dtype,
    load_in_4bit=load_in_4bit, # Will load the 4Bit Quantized Model
)

==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

### 4. 🧠 Model Configuration & Rationale

This section details the specific pre-trained model and the configuration used for loading it. These choices are critical for balancing performance and resource limitations.

### Base Model

* **Model:** `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`
* **Reason for Use:**
    1.  **`unsloth`:** This is a special version of the Llama model, pre-optimized by Unsloth for faster training and lower memory use.
    2.  **`Llama-3.2-3B`:** This model (3 Billion parameters) is small but powerful. It's an excellent balance between being "smart" enough to understand academic rules and "light" enough to be fine-tuned quickly.
    3.  **`Instruct`:** This variant has already been trained to follow instructions, making it much easier to teach our specific "advisor" task.
    4.  **`bnb-4bit`:** This signifies that the model is already quantized (shrunk) to 4-bit, which is a key part of our memory-saving strategy.


## ⚡5. Model Fine-Tuning: PEFT / LoRA

We are not retraining the entire 3-billion-parameter model. That would be computationally impossible on consumer hardware. Instead, we use a technique called **PEFT** (Parameter-Efficient Fine-Tuning), specifically **LoRA** (Low-Rank Adaptation).

This command, `FastLanguageModel.get_peft_model`, freezes the original model's weights and injects tiny, trainable "adapter" layers. This is the core of our strategy: we are fine-tuning a model that has been loaded in 4-bit precision (known as **QLoRA**).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, # A higher alpha value assigns more weight to the LoRA activations
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none", # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.11.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Rationale: Parameter Breakdown
> `r = 16`

  <ol> • What it is: This is the rank (r), or the size, of the LoRA adapters. It determines how many new, trainable parameters we are adding.</ol>

  <ol> • Reason for Use: This is a "sweet spot" value. A higher rank (like 64) would add more parameters, allowing the model to learn more complex patterns but at the cost of slower training and more memory. r = 16 provides a perfect balance of performance and efficiency for this task. </ol>


---


> `target_modules = [...]`

<ol> • What it is: This list tells LoRA where to attach its adapters. The module names (q_proj, k_proj, v_proj, etc.) refer to the most critical parts of the model's architecture, the attention mechanisms and feed-forward networks. </ol>

<ol> • Reason for Use: By targeting all key linear layers, we are giving the model the maximum opportunity to adapt. This "full-target" approach ensures our new knowledge (the study plans) can influence every major component of the model's "thinking" process.</ol>


---


> `lora_alpha = 16`

<ol> • What it is: This is a scaling factor for the adapter's outputs.</ol>

<ol> • Reason for Use: A common practice is to set lora_alpha equal to the r. This normalizes the adapter's influence, preventing the new LoRA weights from having an output that is too large or too small, which leads to a more stable training process.</ol>


---


> `use_gradient_checkpointing = "unsloth"`

<ol> • What it is: This is a powerful memory-saving technique.</ol>

<ol> • Reason for Use: Instead of storing all intermediate calculations (which consumes a huge amount of VRAM), checkpointing saves memory by re-computing them during the backward pass. This trades a small amount of compute time for a massive reduction in memory. We use the "unsloth" version because it is heavily optimized and much faster than the standard True implementation, which is essential for training with our max_sequence of 2048.</ol>


---

> `lora_dropout = 0 & bias = "none"`

<ol> • What they are: Optimization settings.</ol>

<ol> • Reason for Use: The Unsloth library is specifically optimized to run at maximum speed when dropout is set to 0 and bias is set to none. We use these settings to gain the best possible training performance.</ol>

## 6. 💬 Data Formatting: The Chat Template

A raw `.jsonl` file is just data. A large language model, especially an instruction-tuned one like Llama 3, requires a very specific, rigid format to understand the difference between a system prompt, a user's question, and its own expected response.


This function, `formatting_prompts_func`, is the most critical data-preparation step. It takes our human-readable JSON object and translates it into the precise string format that Llama 3 was trained on.

In [ ]:
def formatting_prompts_func(example):
    """
    This function formats a single example from our .jsonl file into the
    Llama 3 chat template.
    """

    # The structure of our .jsonl file
    system_prompt = example.get("system_prompt", None)
    conversation = example.get("conversation", [])

    # Create the 'messages' list for the chat template
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    # Add user and assistant messages
    if len(conversation) == 2:
        messages.append({"role": "user", "content": conversation[0]["content"]})
        messages.append({"role": "assistant", "content": conversation[1]["content"]})
    else:
        # Handle other cases or skip if data is malformed
        return { "text": "" } # Return empty text to skip this example

    # This is the magic function!
    # It applies the full Llama 3 template, including:
    # <|begin_of_text|><|start_header_id|>system<|end_header_id|>...
    # ...and the <|eot_id|> token at the end.
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False, # We return a string, SFTTrainer will tokenize it
        add_generation_prompt=False, # We want the full convo, not just the prompt
    )

    return { "text": formatted_text }



### **Rationale: Why This Function?**

<ol> • Models Require a Specific Format: Llama 3 uses special control tokens (like <|begin_of_text|>, <|start_header_id|>, <|eot_id|>, etc.) to structure a conversation. If we fail to use this exact format, the model will be confused and its performance will be extremely poor.</ol>

<ol>• tokenizer.apply_chat_template: This is the reason the process works. This command is the official, built-in method for formatting. It knows the exact Llama 3 template, so we don't have to build this complex string by hand (which would be very error-prone).</ol>


> `tokenize=False`

<ol>• Reason for Use: We configure this to return a plain string, not token IDs. The SFTTrainer is optimized to receive a dataset of strings and will handle the tokenization itself in a more efficient, batched process.</ol>


> `add_generation_prompt=False`

<ol>• Reason for Use: This is critical for training.</ol>

<ol>• If this were True, the template would stop after the user's question (e.g., ...<|start_header_id|>assistant<|end_header_id|>), prompting the model to generate a new response.

Because this is False, it includes the entire conversation, including the assistant's "correct" answer from our dataset. This is what allows the model to learn: "When you see this user input, you should produce this assistant output."
</ol>

> `return { "text": formatted_text }`

<ol>• Reason for Use: The SFTTrainer (which we will define next) is, by default, configured to look for a column in the dataset named "text". This line creates that exact column, making our dataset plug-and-play with the trainer.</ol>

## 7. 💾 Data Loading & Preparation

Before the model can be trained, the raw `jsonl` data must be loaded into memory and processed into the chat format we defined in the previous step. This is handled in an efficient, "streaming" pipeline.

In [ ]:
# Load your generated dataset
dataset = load_dataset(
    "json",
    data_files="training_data_generated.jsonl",
    split="train",
    streaming=True)

# Apply the formatting function
dataset = dataset.map(formatting_prompts_func, batched=False)

In [ ]:
for row in dataset:
    print(row)
    break

{'system_prompt': "You are a helpful university academic advisor. Your task is to check a student's course history against the university's study plan and prerequisite rules, then provide clear and accurate recommendations. Be polite and specific.", 'context': {'major': 'AI', 'courses_taken': []}, 'conversation': [{'role': 'user', 'content': "Hi, I'm a new AI student. What should I take in my first semester?"}, {'role': 'assistant', 'content': 'Welcome! As a new AI student, you can start with courses that have no prerequisites. I recommend **401099: Computer Skill (1)** and **404101: Calculus (1) **. Both are requirements for your plan.'}], 'text': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are a helpful university academic advisor. Your task is to check a student's course history against the university's study plan and prerequisite rules, then provide clear and accurate recommendations. Be poli

### Rationale: Why This Approach?

* **`load_dataset("json", ...)`**

   * What it is: This is the standard Hugging Face command to load a JSON or .jsonl file. We specify the data_files and assign the entire dataset to the split="train".
   * Reason for Use: This is the simplest, most direct way to load our custom data file into the datasets ecosystem.



* **`streaming=True`**

   * What it is: This is a crucial efficiency flag. Instead of loading the entire file into memory at once, this creates an IterableDataset.

  *   Reason for Use: This saves a significant amount of RAM and time. The data is loaded "lazily," meaning we pull each example from the file only when the trainer needs it. This is highly recommended for large datasets and makes development (like starting a training run) much faster.

* **.map(formatting_prompts_func, ...)`**

   * What it is: This command applies our formatting_prompts_func to every single example in the dataset as it's loaded.
   * Reason for Use: This is the "pipeline" part. It connects our data source (the .jsonl file) to our formatting function. The result is a "stream" of data that is perfectly formatted for the Llama 3 model, ready to be sent to the trainer.

* **`batched=False`**

    * What it is: This setting tells the .map() function to send examples to our function one-by-one.

  * Reason for Use: Our formatting_prompts_func is written to accept a single example (a Python dictionary). batched=False is the correct setting for this logic. It makes the code simpler to read and debug, as we are processing one student record at a time.

##  8. 🚀 The Training Process

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_sequence,
    dataset_num_proc = 2, # Number of processors to use for processing the dataset
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2, # The batch size per GPU/TPU core
        gradient_accumulation_steps = 4, # Number of steps to perform befor each gradient accumulation
        warmup_steps = 5, # Few updates with low learning rate before actual training
        max_steps = 60, # Specifies the total number of training steps (batches) to run.
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc for observability
    ),
)

trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 480 | Num Epochs = 9,223,372,036,854,775,807 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,3.894100
2,3.869000
3,3.717100
4,3.404200
5,3.220000
6,2.873000
7,2.683500
8,2.244900
9,1.941400
10,1.728300


This is the final stage where we bring all components together. The `SFTTrainer` from the `trl` library is a high-level tool that automates the entire training loop. We configure it with our model, dataset, and a set of `TrainingArguments` that define *how* the training is performed.

### SFTTrainer Configuration

The `SFTTrainer` is the high-level wrapper that manages the training.

* **`model = model`**:
    * **What it is:** Passes in the 4-bit, LoRA-configured model we prepared.
    * **Reason for Use:** This is the actual model object that will be trained.
* **`tokenizer = tokenizer`**:
    * **What it is:** Provides the tokenizer that corresponds to the model.
    * **Reason for Use:** The trainer needs this to process data on the fly.
* **`train_dataset = dataset`**:
    * **What it is:** Provides the streaming, formatted dataset.
    * **Reason for Use:** This is the source of all our training examples.
* **`dataset_text_field = "text"`**
    * **What it is:** The name of the column in the dataset that contains the formatted text.
    * **Reason for Use:** This is the critical link to our data preparation. It explicitly tells the `SFTTrainer` to find the formatted, model-ready strings in the `"text"` column that our `formatting_prompts_func` created.
* **`max_seq_length = max_sequence`**:
    * **What it is:** The maximum token length (2048) for any single training example.
    * **Reason for Use:** This ensures all examples are truncated or padded to a uniform length, which is required for batching.
* **`dataset_num_proc = 2`**
    * **What it is:** The number of CPU cores to use for processing the dataset.
    * **Reason for Use:** This speeds up data preparation by using 2 CPU processes in parallel to run the `formatting_prompts_func` on the fly.
* **`packing = False`**
    * **What it is:** A setting that controls whether to "pack" multiple short examples into one sequence.
    * **Reason for Use:** "Packing" is a technique that stuffs multiple short text examples into a single training sequence. Since our student records and recommendations are relatively long, we disable this. Setting this to `False` ensures that each "student query" is treated as one complete, independent training example.

### TrainingArguments (Hyperparameters)

The `TrainingArguments` class defines the specific rules and settings for the training run.

#### Batching & Step Configuration
* **`per_device_train_batch_size = 2`**
    * **What it is:** The number of training examples to process on a single GPU at one time.
    * **Reason for Use:** This is a **memory-saving** setting. It instructs the GPU to only load 2 training examples at a time. `2` is a safe, low value perfect for consumer hardware.
* **`gradient_accumulation_steps = 4`**
    * **What it is:** The number of "mini-batches" to process before performing a single model update.
    * **Reason for Use:** This is the **key to making our small batch size work**. The trainer will process 4 "mini-batches" of 2 (i.e., `2 * 4 = 8` examples) *before* it updates the model's weights. This achieves the stability of a larger "effective batch size" of 8 without the high memory cost.
* **`max_steps = 60`**
    * **What it is:** The total number of model updates (batches) to run before stopping.
    * **Reason for Use:** This tells the trainer to stop after exactly 60 updates. This is used for a **rapid development loop** to quickly see if the model is learning, allowing for fast testing.

#### Learning Rate & Optimization
* **`learning_rate = 2e-4`**
    * **What it is:** The step size for the optimizer, controlling how much the model's weights are adjusted during each update.
    * **Reason for Use:** This is the "speed" at which the model learns. `2e-4` (or 0.0002) is a well-established, effective learning rate for LoRA fine-tuning.
* **`lr_scheduler_type = "linear"`**
    * **What it is:** The strategy used to change the learning rate *during* the training run.
    * **Reason for Use:** This makes the learning rate start at `2e-4` and slowly, linearly decrease down to `0` by the `max_steps`. The model learns quickly at the start and then makes smaller, more precise adjustments at the end.
* **`warmup_steps = 5`**
    * **What it is:** The number of initial steps where the learning rate gradually increases from 0 to the `learning_rate`.
    * **Reason for Use:** This is a stabilization technique. For the first 5 steps, the trainer uses a *very* low learning rate and gradually increases it. This "warms up" the model and prevents unstable updates at the start.
* **`optim = "adamw_8bit"`**
    * **What it is:** The specific optimizer algorithm used to calculate the model's weight updates.
    * **Reason for Use:** This is another critical **memory-saving** setting. The "optimizer" (which calculates model updates) traditionally stores its data in 32-bit. This 8-bit version drastically reduces that memory overhead, preventing "Out of Memory" errors.
* **`weight_decay = 0.01`**
    * **What it is:** A regularization technique that adds a small penalty to large model weights.
    * **Reason for Use:** This is a best-practice setting. It helps prevent "overfitting" and ensures the model learns general patterns instead of just memorizing the data.

#### Performance & Precision
* **`fp16 = not is_bfloat16_supported()`**
* **`bf16 = is_bfloat16_supported()`**
    * **What it is:** Settings that enable 16-bit mixed-precision training. `fp16` (float16) and `bf16` (bfloat16) are two different 16-bit formats.
    * **Reason for Use:** This is a massive **speed and memory optimization**. It tells the trainer to use 16-bit floating-point numbers for most calculations instead of the full 32-bit. The code automatically selects the best available 16-bit format, cutting VRAM usage nearly in half.

#### Logging & Reproducibility
* **`logging_steps = 1`**
    * **What it is:** How often (in steps) to print training information (like the loss) to the console.
    * **Reason for Use:** This gives us **maximum visibility** by logging the training loss after *every single step*. This is perfect for a short run (like 60 steps) as we can watch the model learn in real-time.
* **`seed = 3407`**
    * **What it is:** A number used to initialize all random processes in the training run.
    * **Reason for Use:** This is **critical for reproducibility**. By setting a specific seed, we guarantee that if we run this script again, we will get the *exact same results*.
* **`output_dir = "outputs"`**
    * **What it is:** The name of the folder where all training results (model checkpoints, logs) will be saved.
    * **Reason for Use:** This is the destination folder where all training results will be saved, keeping the project organized.
* **`report_to = "none"`**
    * **What it is:** A setting that controls where to send training logs (e.g., "wandb", "tensorboard").
    * **Reason for Use:** Setting this to `"none"` disables reporting to third-party services like "Weights & Biases." It keeps the training run simple, local, and self-contained.

## 9. 🤖 Using the Model (Inference)

After the model is trained, we need to configure it for "inference" (generating answers). This process involves preparing a prompt, optimizing the model for speed, and decoding the final response.

In [ ]:
# --- 1. Define Your Test Samples ---
# This system prompt MUST match the one you used in training
system_prompt = "You are a helpful university academic advisor. Your task is to check a student's course history against the university's study plan and prerequisite rules. When a student asks for course recommendations, you must provide at least 4 mandatory subjects. Always be polite and specific."

# Create a list of all user questions you want to test
test_prompts = [
    "Hi, I'm a new AI student. What should I take in my first semester?",
    "I am an AI student. I have taken 'Calculus (1)' and 'Computer Skill (1)'. What's next?",
    "I'm a Computer Science student, not AI. I have taken 401099 and 404101. What should I register for?",
    "What are the prerequisites for 'Data Structures'?"
]

# This list will store all the model's answers
model_answers = []

# Get the special end-of-text token ID for Llama 3
# We define this once, outside the loop
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("--- Starting Batch Inference ---")

# --- 2. Loop Through and Generate Answers ---
for user_prompt in test_prompts:
    print(f"Processing prompt: '{user_prompt[:50]}...'")

    # Format the prompt for Llama 3
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    # Use apply_chat_template for inference
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Key for inference
        return_tensors = "pt",
    ).to("cuda") # Move inputs to GPU

    # Run the model
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        eos_token_id=terminators, # Stop generating at <|eot_id|>
    )

    # Decode *only* the new tokens and save the answer
    response_text = tokenizer.decode(
        outputs[0][inputs.shape[-1]:], # Slice to get only the new tokens
        skip_special_tokens=True
    )

    # Add the clean answer to our list
    model_answers.append(response_text)

print("\n--- ✅ Inference Complete ---")


# --- 3. Print All Saved Answers ---
print("\n--- 📋 All Model Answers ---")
for i, (question, answer) in enumerate(zip(test_prompts, model_answers)):
    print(f"\n--- [Sample {i+1}] ---")
    print(f"🤔 USER: {question}")
    print(f"🤖 MODEL: {answer}")
    print("--------------------")

--- Starting Batch Inference ---
Processing prompt: 'Hi, I'm a new AI student. What should I take in my...'
Processing prompt: 'I am an AI student. I have taken 'Calculus (1)' an...'
Processing prompt: 'I'm a Computer Science student, not AI. I have tak...'
Processing prompt: 'What are the prerequisites for 'Data Structures'?...'

--- ✅ Inference Complete ---

--- 📋 All Model Answers ---

--- [Sample 1] ---
🤔 USER: Hi, I'm a new AI student. What should I take in my first semester?
🤖 MODEL: Welcome to the university! As a new AI student, I recommend the following four mandatory courses for your first semester:

1. **401102: Introduction to Artificial Intelligence** (1 semester) - This course provides a solid foundation in AI, including its history, applications, and fundamental concepts.
2. **401101: Programming Languages and Data Structures** (1 semester) - This prerequisite course is necessary for all AI-related courses, ensuring you have a solid grasp of programming concepts and data

* **`Sys_prompt`**

    * **What it is:** A "meta-prompt" that defines the persona of the assistant.

    * **Reason for Use:** By defining a clear persona, we can control how the model responds. In this case, we are instructing it to be "reflective" and "thorough."

* **`test_prompts`**

    * **What it is:** What it is: A Python list of different questions.

    * **Reason for Use:** This allows us to validate the model's performance across multiple scenarios (a new student, a student in progress, a student from a different major, a specific question) all at once.


* **`terminators`**
    * **What it is:** A list containing the token IDs for the two "end" tokens.

    * **Reason for Use:** This is a critical instruction for `model.generate()`. It tells the model to stop generating as soon as it outputs either the standard End-of-Sequence token (`eos_token_id`) or the Llama 3-specific End-of-Turn token (`<|eot_id|>`). This prevents the model from rambling or generating a fake user response.

### 2. Loop Through and Generate Answers
This `for` loop is the core of the batch process. It iterates through each `user_prompt`, formats it, runs the model, and saves the decoded answer.

* **`inputs = tokenizer.apply_chat_template(...)`**
    * **What it is:** The tokenizer function converting the messages list into tensors.

    * **Reason for Use:** The `add_generation_prompt = True` flag is essential for inference. It formats the prompt (e.g., <|start_header_id|>system...<|start_header_id|>user...<|start_header_id|>assistant<|end_header_id|>) and stops exactly at the point where the assistant should start talking, prompting it to generate a response.

* **`outputs = model.generate(...)`**
    * **What it is:** The core function that runs the model.

    * **Reason for Use:**
        * **`do_sample=True` & `temperature=0.7`:** This combination enables sampling, which produces more natural, less robotic answers. A temperature of `0.7` is a good balance between creativity and factuality.
        * **`eos_token_id=terminators:`** This ensures the model stops generating text at the correct place, using the list we defined earlier.

* **`response_text = tokenizer.decode(outputs[0][inputs.shape[-1]:]...)`**
    * **What it is:** Decodes the output tokens back into a text string.
    * **Reason for Use:** The slice `[inputs.shape[-1]:]` is the most important part. outputs[0] contains the entire conversation (input prompt + new answer). This slice selects only the newly generated tokens that come after the input, giving us just the model's clean response..
    * **`skip_special_tokens=True:`** This removes any `eos` tokens from the final text, making it clean.



### 3. Print All Saved Answers

* **`for ... in enumerate(zip(...))`**

    * **What it is:** A Python function that loops through both the test_prompts and model_answers lists at the same time.
    * **Reason for Use:** This provides a clean, side-by-side comparison, allowing you to quickly and easily validate how the model responded to each specific test prompt.